In [ ]:
!pip install --upgrade transformers peft accelerate -q

In [1]:


import os
import gc
import cv2
import torch
import random
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, random_split
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import xml.etree.ElementTree as ET


def extract_text_crops(image_path, xml_path):
    """
    Returns list of (image_path, xmin, ymin, xmax, ymax, label) tuples.
    Images are NOT loaded into memory here — only paths and coordinates.
    """
    if not Path(image_path).exists():
        return []

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return []

    samples = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        if name_el is None or name_el.text != 'text':
            continue

        text_el = obj.find('text')
        if text_el is None or not text_el.text:
            continue

        ground_truth = text_el.text.strip()
        if not ground_truth:
            continue

        bndbox = obj.find('bndbox')
        try:
            xmin = int(bndbox.find('xmin').text)
            ymin = int(bndbox.find('ymin').text)
            xmax = int(bndbox.find('xmax').text)
            ymax = int(bndbox.find('ymax').text)
        except (ValueError, AttributeError):
            continue

        samples.append((str(image_path), xmin, ymin, xmax, ymax, ground_truth))

    return samples



class SchematicTextDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, xmin, ymin, xmax, ymax, label = self.samples[idx]

        img = cv2.imread(image_path)
        if img is None:
            # Return a blank sample if image fails to load
            img = np.zeros((32, 32, 3), dtype=np.uint8)

        pad = 5
        h, w = img.shape[:2]
        xmin_p = max(0, xmin - pad)
        ymin_p = max(0, ymin - pad)
        xmax_p = min(w, xmax + pad)
        ymax_p = min(h, ymax + pad)

        crop = img[ymin_p:ymax_p, xmin_p:xmax_p]
        if crop.size == 0:
            crop = np.zeros((32, 32, 3), dtype=np.uint8)

        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb).convert("RGB")

        pixel_values = self.processor(
            pil_img, return_tensors="pt"
        ).pixel_values.squeeze()

        encoding = self.processor.tokenizer(
            label,
            padding="max_length",
            max_length=32,
            truncation=True,
        )
        labels = encoding.input_ids

        # Replace padding token id with -100 so loss ignores padding
        pad_id = self.processor.tokenizer.pad_token_id
        labels = [l if l != pad_id else -100 for l in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels, dtype=torch.long),
        }



print("Scanning dataset for text annotations...")

DATASET_ROOT = Path('/kaggle/input/datasets/johannesbayer/cghd1152')

all_samples = []
xml_files = list(DATASET_ROOT.rglob('*.xml'))
print(f"Found {len(xml_files)} XML files")

for xml_file in xml_files:
    image_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.jpg')
    if not image_file.exists():
        image_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.png')
    if not image_file.exists():
        continue
    crops = extract_text_crops(image_file, xml_file)
    all_samples.extend(crops)

print(f"Total text crop samples: {len(all_samples)}")

random.shuffle(all_samples)

# Sanity check
for image_path, xmin, ymin, xmax, ymax, label in all_samples[:10]:
    print(f"  Label: '{label}'")


print("\nLoading TrOCR model...")
MODEL_CHECKPOINT = 'microsoft/trocr-small-handwritten'

processor = TrOCRProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_CHECKPOINT)

# Decoder config — only non-generation parameters go in model.config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id

# Generation parameters go in generation_config
model.generation_config.max_length = 32
model.generation_config.early_stopping = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty = 2.0
model.generation_config.num_beams = 4


train_size = int(0.8 * len(all_samples))
val_size = len(all_samples) - train_size
train_samples, val_samples = random_split(all_samples, [train_size, val_size])

train_dataset = SchematicTextDataset(list(train_samples), processor)
val_dataset = SchematicTextDataset(list(val_samples), processor)

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")

OUTPUT_DIR = '/kaggle/working/trocr-schematic'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
)


gc.collect()
torch.cuda.empty_cache()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\nStarting training...")
trainer.train()


FINAL_MODEL_DIR = '/kaggle/working/trocr-schematic-final'
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

print(f"Model saved to {FINAL_MODEL_DIR}")


Scanning dataset for text annotations...
Found 3269 XML files
Total text crop samples: 59206
  Label: '12V'
  Label: 'C2-'
  Label: '14'
  Label: 'Q3'
  Label: '9'
  Label: 'R'
  Label: 'R3'
  Label: 'C26'
  Label: '3'
  Label: '25'

Loading TrOCR model...


The image processor of type `DeiTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train: 47364  |  Val: 11842

Starting training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /kaggle/working/trocr-schematic-final


In [ ]:
pip install jiwer

In [ ]:
import os
import torch
import random
import jiwer
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# 1. Same extraction logic used in your training script
def extract_text_crops(image_path, xml_path):
    if not Path(image_path).exists(): return []
    try: tree = ET.parse(xml_path)
    except ET.ParseError: return []
    root = tree.getroot()
    samples = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        text_el = obj.find('text')
        if name_el is None or name_el.text != 'text' or text_el is None or not text_el.text.strip():
            continue
        ground_truth = text_el.text.strip()
        bndbox = obj.find('bndbox')
        try:
            xmin, ymin = int(bndbox.find('xmin').text), int(bndbox.find('ymin').text)
            xmax, ymax = int(bndbox.find('xmax').text), int(bndbox.find('ymax').text)
            samples.append((str(image_path), xmin, ymin, xmax, ymax, ground_truth))
        except (ValueError, AttributeError): continue
    return samples

# 2. Setup paths (Update these if running locally instead of Kaggle)
DATASET_ROOT = Path('/kaggle/input/datasets/johannesbayer/cghd1152')
MODEL_DIR = '/kaggle/input/models/amtoj5/trocrschematicfinal/transformers/default/1/trocrSchematicFinal'

# 3. Load dataset and recreate the split
print("Scanning dataset...")
all_samples = []
for xml_file in DATASET_ROOT.rglob('*.xml'):
    img_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.jpg')
    if not img_file.exists(): img_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.png')
    if img_file.exists():
        all_samples.extend(extract_text_crops(img_file, xml_file))

# Use a fixed seed if you want to reproduce the exact same split later
random.seed(42) 
random.shuffle(all_samples)
train_size = int(0.8 * len(all_samples))
val_samples = all_samples[train_size:]
print(f"Evaluating on {len(val_samples)} validation samples...")

# 4. Load your trained model
print("Loading model...")
processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 5. Evaluation Loop
exact_matches = 0
total_cer = 0.0
valid_samples = 0

print("Running evaluation...")
for img_path, xmin, ymin, xmax, ymax, ground_truth in tqdm(val_samples):
    # Crop logic
    img = cv2.imread(img_path)
    if img is None: continue
    pad = 5
    h, w = img.shape[:2]
    crop = img[max(0, ymin-pad):min(h, ymax+pad), max(0, xmin-pad):min(w, xmax+pad)]
    if crop.size == 0: continue
    
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb).convert("RGB")
    
    # Generate prediction
    pixel_values = processor(pil_img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    
    prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    # Exact Match
    if prediction == ground_truth:
        exact_matches += 1
        
    # Character Error Rate (CER)
    # jiwer throws an error if ground truth is completely empty, so we handle it
    if len(ground_truth) > 0:
        cer = jiwer.cer(ground_truth, prediction)
        total_cer += cer
        valid_samples += 1

# 6. Calculate Final Metrics
exact_match_accuracy = (exact_matches / len(val_samples)) * 100
average_cer = total_cer / valid_samples
character_accuracy = max(0, (1 - average_cer) * 100) # Floor at 0%

print("\n--- FINAL METRICS FOR REPORT ---")
print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")
print(f"Character Accuracy:   {character_accuracy:.2f}%")
print(f"(Average CER: {average_cer:.4f})")